In [12]:
import nibabel as nib
import numpy as np
from helper import normalized_modality, format_index, get_filepath, pad_to_256
from torch.utils.data import random_split, DataLoader, Dataset
import torch.optim as optim
import torch.nn.functional as F
from monai.losses import DiceCELoss
from monai.transforms import (
    RandFlipd, RandRotate90d, RandAffined,
    RandGaussianNoised, RandAdjustContrastd,
    Compose
)
import torch.nn as nn 
import torch
import wandb
from pathlib import Path
from dotenv import dotenv_values, find_dotenv

config = dotenv_values(find_dotenv(usecwd=True))
TR_DATA_PATH = Path(config.get("TR_DATA_PATH"))
wandb.login(config.get("WANDB_API_KEY"));

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\kacpe\_netrc


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [14]:
BRAINS = 120

class CustomDataset(Dataset):
    def __init__(self, path):
        super().__init__()
        self.path = path
        self.dirnames = [d for d in Path(path).iterdir() if d.is_dir()]
        self.len = BRAINS
        # self.len = len(self.dirnames) # 369 brains
        self.slices = 155 # slices per brain

    def __len__(self):
        return self.len
    
    def get_path(self, brain_index, mod):
        brain_index += 1
        return get_filepath(brain_index, mod)
    
    def get_brain_mod(self, index, mod):
        path = self.get_path(index, mod)
        img = nib.load(path)
        brain = torch.tensor(img.get_fdata(caching = 'unchanged'), dtype=torch.float32)
        brain = brain.permute(2, 0, 1)

        if mod != "seg":
            brain = normalized_modality(brain)
            brain = pad_to_256(brain)
            brain = brain.unsqueeze(1)
        
        if mod == "seg":
            brain[brain == 4] = 3
            brain = pad_to_256(brain)
            brain = brain.to(torch.int64)

        return brain
            
    def __getitem__(self, index):
        # t1 = self.get_brain_mod(index, "t1")
        t1ce = self.get_brain_mod(index, "t1ce")
        # t2 = self.get_brain_mod(index, "t2")
        flair = self.get_brain_mod(index, "flair")
        seg = self.get_brain_mod(index, "seg")
        brain = torch.cat([flair, t1ce], dim=1)
        return brain, seg
        

In [15]:
train_dataset = CustomDataset(TR_DATA_PATH)

train_size = int(len(train_dataset) * 0.9)
val_size = len(train_dataset) - train_size

train_sub, val_sub = random_split(train_dataset, [train_size, val_size])
train_loader = DataLoader(train_sub, batch_size=None, shuffle=True)

val_loader = DataLoader(val_sub, batch_size=None, shuffle=False)

In [16]:
CLASS_NAMES = ["background", "necrotic", "edema", "enhancing"]

In [17]:
def dice_per_class(logits, y, num_classes=4, eps=1e-6):
    preds = logits.argmax(dim=1)
    scores = {}

    for c in range(num_classes):
        pred_c = (preds == c).float()
        true_c = (y == c).float()

        intersection = (pred_c * true_c).sum()
        denom = pred_c.sum() + true_c.sum()

        # if a class is absent in both pred and target, score is 1 (perfect)
        if denom < eps:
            scores[CLASS_NAMES[c]] = 1.0
        else:
            scores[CLASS_NAMES[c]] = (2.0 * intersection / denom).item()

    return scores

In [18]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    cumulative_dice = {c: 0.0 for c in CLASS_NAMES}
    num_batches = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y.unsqueeze(1)) # needed for loss function

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

        batch_dice = dice_per_class(logits, y)
        for c in CLASS_NAMES:
            cumulative_dice[c] += batch_dice[c]
        num_batches += 1

    avg_dice = {c: cumulative_dice[c] / num_batches for c in CLASS_NAMES}
    return total_loss / total_samples, avg_dice

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    cumulative_dice = {c: 0.0 for c in CLASS_NAMES}
    num_batches = 0

    for x, y in loader:
        step = 20
        for i in range(0, x.shape[0], step):
            xb = x[i:i+step].to(device)
            yb = y[i:i+step].to(device)
            yb_input = yb.unsqueeze(1) # this is needed for the loss function

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb_input)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)
            total_samples += xb.size(0)

            batch_dice = dice_per_class(logits.detach(), yb)
            for c in CLASS_NAMES:
                cumulative_dice[c] += batch_dice[c]
            num_batches += 1

        del x,y
        
    avg_dice = {c: cumulative_dice[c] / num_batches for c in CLASS_NAMES}
    return total_loss / total_samples, avg_dice

In [20]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # encoding
        self.econv1 = nn.Conv2d(2, 32, 3, padding=1) # change the input dimensions to match the number of modalities
        self.econv2 = nn.Conv2d(32, 32, 3, padding=1)
        
        self.econv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.econv4 = nn.Conv2d(64, 64, 3, padding=1)
        self.max2 = nn.MaxPool2d(2, 2)
        
        self.econv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.econv6 = nn.Conv2d(128, 128, 3, padding=1)
        self.max3 = nn.MaxPool2d(2, 2)
        
        self.econv7 = nn.Conv2d(128, 256, 3, padding=1)
        self.econv8 = nn.Conv2d(256, 256, 3, padding=1)

        self.econv9 = nn.Conv2d(256, 512, 3, padding=1)

        # decoding
        self.dconv1 = nn.Conv2d(512, 512, 3, padding=1)

        self.upconv1 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dconv2 = nn.Conv2d(512, 256, 3, padding=1)
        self.dconv3 = nn.Conv2d(256, 256, 3, padding=1)
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dconv4 = nn.Conv2d(256, 128, 3, padding=1)
        self.dconv5 = nn.Conv2d(128, 128, 3, padding=1)
        
        self.upconv3 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dconv6 = nn.Conv2d(128, 64, 3, padding=1)
        self.dconv7 = nn.Conv2d(64, 64, 3, padding=1)

        self.upconv4 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dconv8 = nn.Conv2d(64, 32, 3, padding=1)
        self.dconv9 = nn.Conv2d(32, 32, 3, padding=1)

        self.dconv10 = nn.Conv2d(32, 4, 1)

        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.relu(self.econv1(x))
        x = self.relu(self.econv2(x))
        x1 = x
        x = self.maxpool(x)

        x = self.relu(self.econv3(x))
        x = self.relu(self.econv4(x))
        x2 = x
        x = self.maxpool(x)
        
        x = self.relu(self.econv5(x))
        x = self.relu(self.econv6(x))
        x3 = x
        x = self.maxpool(x)

        x = self.relu(self.econv7(x))
        x = self.relu(self.econv8(x))
        x4 = x
        x = self.maxpool(x)

        x = self.relu(self.econv9(x))

        x = self.relu(self.dconv1(x))

        x = self.upconv1(x)
        x = self.relu(self.dconv2(torch.cat([x4, x], dim=1)))
        x = self.relu(self.dconv3(x))

        x = self.upconv2(x)
        x = self.relu(self.dconv4(torch.cat([x3, x], dim=1)))
        x = self.relu(self.dconv5(x))

        x = self.upconv3(x)
        x = self.relu(self.dconv6(torch.cat([x2, x], dim=1)))
        x = self.relu(self.dconv7(x))

        x = self.upconv4(x)
        x = self.relu(self.dconv8(torch.cat([x1, x], dim=1)))
        x = self.relu(self.dconv9(x))

        x = self.dconv10(x)
        return x

In [21]:
model = UNet()
model = model.to(device)

In [22]:
LR = 5e-4
EPOCHS = 35

optimizer = optim.Adam(model.parameters(), lr = LR)

# criterion = DiceFocalLoss(
#     to_onehot_y=True,
#     softmax=True,
#     squared_pred=True,
#     gamma=2.0,       # focal strength, the higher it is the more it focuses on hard examples
#     lambda_dice=0.5, # balance between Dice and Focal terms
#     lambda_focal=0.5,
# )

class_weights = torch.tensor([0.1, 2.0, 2.0, 2.0], device=device)
criterion = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    squared_pred=True,
    weight=class_weights,
)

run = wandb.init(
    entity="kappo-university-of-wroc-aw",
    project="Brain-tumor-segmentation",
    config={
        "learning_rate": LR,
        "architecture": "UNET",
        "dataset": "BraTS2020",
        "loss" : "weighted DiceCELoss",
        "loss ratio" : "50/50", # CHANGE THIS TO MATCH THE DICE/ENTOPY RATIO IN THE LOSS FUNCTION WHEN ITS ADDED 
        "weights" : class_weights,
        "epochs": EPOCHS,
        "no. brains": BRAINS
    },
)

for epoch in range(EPOCHS):
    tr_loss, tr_dice = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_dice = evaluate(model, val_loader, criterion, device)

    log_dict = {
        "epoch": epoch + 1,
        "train_loss": tr_loss,
        "val_loss": val_loss,
    }
    
    for c in CLASS_NAMES:
        log_dict[f"train_dice/{c}"] = tr_dice[c]
        log_dict[f"val_dice/{c}"]   = val_dice[c]

    run.log(log_dict)

    print(
        f"Epoch {epoch+1} | "
        f"train_loss={tr_loss:.4f} | val_loss={val_loss:.4f}\n"
        f"  Train Dice: { {k: f'{v:.3f}' for k,v in tr_dice.items()} }\n"
        f"  Val   Dice: { {k: f'{v:.3f}' for k,v in val_dice.items()} }"
    )

run.finish()

Epoch 1 | train_loss=1.9523 | val_loss=1.3446
  Train Dice: {'background': '0.996', 'necrotic': '0.577', 'edema': '0.439', 'enhancing': '0.490'}
  Val   Dice: {'background': '0.995', 'necrotic': '0.107', 'edema': '0.295', 'enhancing': '0.544'}
Epoch 2 | train_loss=1.3618 | val_loss=1.4071
  Train Dice: {'background': '0.997', 'necrotic': '0.454', 'edema': '0.454', 'enhancing': '0.491'}
  Val   Dice: {'background': '0.996', 'necrotic': '0.200', 'edema': '0.319', 'enhancing': '0.549'}
Epoch 3 | train_loss=1.3394 | val_loss=1.3722
  Train Dice: {'background': '0.997', 'necrotic': '0.478', 'edema': '0.476', 'enhancing': '0.520'}
  Val   Dice: {'background': '0.997', 'necrotic': '0.301', 'edema': '0.363', 'enhancing': '0.580'}
Epoch 4 | train_loss=0.9966 | val_loss=0.8456
  Train Dice: {'background': '0.997', 'necrotic': '0.631', 'edema': '0.465', 'enhancing': '0.668'}
  Val   Dice: {'background': '0.998', 'necrotic': '0.313', 'edema': '0.498', 'enhancing': '0.655'}
Epoch 5 | train_loss=1.1

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 35 | train_loss=0.4772 | val_loss=0.5269
  Train Dice: {'background': '0.999', 'necrotic': '0.750', 'edema': '0.621', 'enhancing': '0.824'}
  Val   Dice: {'background': '0.999', 'necrotic': '0.492', 'edema': '0.588', 'enhancing': '0.710'}


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train_dice/background,▃▅▅▄▁▃▆▆▇▆▇▇▇▇█████▇███████████▇███
train_dice/edema,▁▂▂▂▁▃▄▄▄▅▅▅▆▆▆▇▆▇▇▇▇█▇██▇█▇███▇█▇█
train_dice/enhancing,▁▁▂▅▅▇▇▇▇▇▇▇█▇▇▇████████▇█████▇████
train_dice/necrotic,▄▁▂▅▄▆▇▇▇▇▇▇██▇█▇██████████████████
train_loss,█▅▅▃▄▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁
val_dice/background,▁▂▄▇▅▃▆▃▆▃▅▆▃▅▇▃█▇▅▆▆██▅▇█▅▆█▆█▆▇▇▇
val_dice/edema,▁▁▂▅▄▂▃▂▄▂▄▅▃▄▆▃▇▆▄▅▅█▇▄▇▇▄▆▇▅▇▅▅▆▇
val_dice/enhancing,▁▁▂▅▅▇▅▇▇▇▆▆▆▆▇▇▇▆▇█▅█▆▇██▇█▇▇█▇▅▇▆
val_dice/necrotic,▁▃▄▄▅▆▅▅▄▆▆▆▇▇▆▇▆▆▇██▇███▆▆▇██▇█▇█▇
+1,...


In [23]:
# checkpoint = {
#     "epoch": 10,
#     "model_state_dict": model.state_dict(),
#     "optimizer_state_dict": optimizer.state_dict(),
#     "loss": tr_loss_list,
#     "acc": tr_acc_list
# }

# torch.save(checkpoint, "unet_checkpoint.pth")

In [24]:
# checkpoint = torch.load("unet_checkpoint.pth", map_location=device)

# tr_loss_list = checkpoint["loss"]
# tr_acc_list = checkpoint["acc"]

In [25]:
# import matplotlib.pyplot as plt

# plt.plot(tr_loss_list)
# plt.xlabel("epoch")
# plt.ylabel("loss")
# plt.show()

# plt.plot(tr_acc_list)
# plt.xlabel("epoch")
# plt.ylabel("acc")
# plt.show()

In [26]:
def aggregate_prediction(pred):
    # may be usefull for simplified segmentation
    whole_tumor = (pred > 0)
    tumor_core = np.isin(pred, [1, 3])
    enhancing = (pred == 3)
    return whole_tumor, tumor_core, enhancing
